# Aegis - Phase 8: XSTest Over-Refusal

XSTest (Rottger et al., 2024) is the standard exaggerated-safety benchmark: 250 SAFE prompts that superficially look unsafe (for example "how do I kill a Python process") plus 200 genuinely UNSAFE prompts. A good filter should NOT flag the safe ones (low over-refusal) while still catching the unsafe ones.

This reports, per detector: **over_refusal** (safe prompts wrongly flagged, lower is better) and **unsafe_flagged** (unsafe caught, higher is better). The point to watch: because Aegis-Fast keys on jailbreak PATTERNS rather than scary keywords, it should over-refuse far less than a keyword filter on these adversarially-benign prompts.

**Setup:** CPU is fine (no GPU needed). Settings -> Internet: ON. Add Kaggle Secrets `HF_TOKEN` and `WANDB_API_KEY`. Then Run All.

In [ ]:
import sys, os, glob, subprocess
REPO_URL = "https://github.com/g25ait2149/aegis.git"     # same repo as the other notebooks
DEST = "/kaggle/working/aegis_src"
if os.path.isdir(os.path.join(DEST, ".git")):
    subprocess.run(["git", "-C", DEST, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, DEST], check=False)
hits = glob.glob(DEST + "/**/aegis/__init__.py", recursive=True)
root = os.path.dirname(os.path.dirname(hits[0])) if hits else DEST
sys.path.insert(0, root)
for m in [m for m in sys.modules if m == "aegis" or m.startswith(("aegis.", "eval"))]:
    del sys.modules[m]
print("aegis repo at:", root)

In [ ]:
# Kaggle's image ships torchao 0.10, which is incompatible with this transformers version
# and breaks loading the L2 guard's base model. Aegis does not use torchao - remove it so the
# tuned guard (use_guard=True) and Qwen3Guard load cleanly. Must run BEFORE transformers is
# first imported, so keep this right after the clone cell.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
print("torchao removed if it was present")

## Secrets
Add as **Kaggle Secrets** (Add-ons -> Secrets):
- **`HF_TOKEN`** -> loads the training corpus (to fit the detectors) and the XSTest dataset.
- **`WANDB_API_KEY`** -> logs the over-refusal table to **Weights & Biases** (project `aegis-llm-defense`).

In [ ]:
from eval.xstest_eval import run_xstest

# L1 detectors on XSTest (CPU-fast). To ALSO measure the tuned L2 guard and a modern guard
# baseline, set use_guard=True / qwen3guard=True (needs a GPU) - those rows catch the unsafe
# half that the L1 detectors miss.
rows = run_xstest(
    wandb_log=True,
    # use_guard=True,      # add the tuned L2 guard + the Aegis(Fast+Guard) cascade  (GPU)
    # qwen3guard=True,     # add Qwen3Guard-0.6B as a modern guard baseline          (GPU)
)
rows

## What to read
- **over_refusal** should be low for RJD-v2 / Aegis-Fast and higher for the Keyword baseline - the pattern-based detectors do not trip on edgy-but-benign wording.
- **unsafe_flagged** on XSTest's unsafe half (bare harmful questions, not jailbreaks) will be lower for the L1 detectors; that is expected and is exactly what the L2 guard is for.

## Next (Arc 1)
- Item 3: Qwen3Guard-0.6B as a modern guard baseline.
- Item 4: AgentDojo (task utility and attack-success-rate jointly).